# CFPB - Customer Complaint Analysis

The **Consumer Financial Protection Bureau** ([CFPB](https://www.consumerfinance.gov/about-us/the-bureau/)) is a U.S. government regulatory agency established to protect consumers in the financial marketplace. It oversees a wide array of financial entities and products, including: Banks and Credit Unions, Payday and Title Lenders, Credit Reporting Agencies, Debt Collectors, Prepaid cards and BNPL services, etc. The agency operates a centralized system where consumers can submit complaints regarding financial institutions, prompting the company to review and respond. 

According to the official 2025 statistics: *the CFPB received around $6.63M$ consumer complaints via its registered portal, toll-free number, postal mail, referrals from other regulatory bodies, etc., and routed around $5.98M$ complaints to over $4K$ companies for review with an expected response TAT of ${<}15days$.* 
[[annual-report_2025](https://www.consumerfinance.gov/data-research/research-reports/2025-consumer-response-annual-report/)].[[annual-report_2024](https://www.consumerfinance.gov/data-research/research-reports/2024-consumer-response-annual-report/)].[[annual-report_2023](https://www.consumerfinance.gov/data-research/research-reports/consumer-response-annual-report-2023/)].[[annual-report_2022](https://www.consumerfinance.gov/data-research/research-reports/2022-consumer-response-annual-report/)].[[annual-report_2021](https://www.consumerfinance.gov/data-research/research-reports/2021-consumer-response-annual-report/)]\
Pursuant to the 'Dodd-Frank Wall Street Reform and Consumer Protection Act of 2010', the bureau submits semi-annual and annual reports directly to Congress and the President. These archived reports serve as a valuable resource for a detailed breakdown of -
- Complaint categories (e.g. - 'credit or consumer reporting' category alone contributes $88\%$ of complaints)
- Demographics of affected customers (e.g. - Georgia has the highest number of complaints per capita among all states)
- Year-over-year growth (e.g. - total complaint volume roughly doubled from $3.19M$ in the previous year)

Hence, rather than reinventing the wheel on EDA, I am moving forward to what matters the most - *the analysis of the vast corpus of text data available.* The portal states: <span style="font-family:'Courier New', monospace; font-style:italic;">"Complaint narratives are consumers’ descriptions of their experiences in their own words"</span>, <span style="font-family:'Courier New', monospace; font-style:italic;">"We do not adopt their views or verify that their experiences are accurate or unbiased"</span>; the CFPB only performs PII masking before uploading the narratives to the database.\
When I came across the dataset on `Kaggle`[[1](https://www.kaggle.com/datasets/sebastienverpile/consumercomplaintsdata)][[2](https://www.kaggle.com/datasets/selener/consumer-complaint-database)] for the first time, I reviewed around $20\text{-}30$ notebooks to gain an overview. Almost all of the attempts involve training a `classifier model` varying from simple `TF-IDF` to advanced `transformer` models. While this is completely acceptable as a toy problem for showcasing programming skills, none of these notebooks addresses the underlying business problem. Consider the CFPB's description of the data collection process: <span style="font-family:'Courier New', monospace; font-style:italic;">"The Consumer Complaint Database shows the consumer's original products, sub-products, issues, and sub-issues selections consistent with the options available on the form at the time the consumer submitted the complaint"</span>. i.e., the data annotation is done by individual customers themselves, not something directly actionable for CFPB staff.\
The most common justification for building classification models is that, *Sometimes customers struggle to choose the correct complaint category. Allowing plain-text submissions and automated tagging will reduce friction in the process.*\
Is that really the case? Let's think through it from a data practitioner's perspective:
- If the difficulty with choosing the correct tag is very prominent, a significant chunk of your training database is likely to be already mislabelled by the customer. The model will fail due to 'garbage in, garbage out'. 
- If the issue is a very rare scenario, it is better to keep the current process as is.
- In the moderate case, suppose you have successfully deployed a classifier today. However, six months down the line, how will you prevent 'model drift'? Remember, you are no longer receiving labelled data; you only have predicted labels. Does the incremental effort required to solve such problems justify the ROI?

<div style="border: 2px solid black; padding: 15px; font-size: 1.2em; font-weight: bold; border-radius: 5px; width: max-content;">
    It is more sensible to spend effort understanding the themes and patterns within tagged complaints than trying to automatically assign those tags.
</div>



In [4]:
import pandas as pd

Source_file = './data.csv'
pd.read_csv(Source_file,
    header=0, index_col=False, dtype=str,
    nrows=5
)

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2026-03-23T17:43:01.000Z,Debt collection,Other debt,Written notification about debt,Notification didn't disclose it was an attempt...,There are collection accounts on my report tha...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TN,37122,NaN,Web,2026-03-23T17:58:04.000Z,Closed with non-monetary relief,Yes,20510057
1,2026-03-23T18:09:14.000Z,Debt collection,Other debt,Written notification about debt,Notification didn't disclose it was an attempt...,There are collection accounts on my report tha...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",OH,44106,NaN,Web,2026-03-23T18:22:09.000Z,Closed with non-monetary relief,Yes,20511052
2,2026-03-24T15:09:00.000Z,Debt collection,Other debt,Written notification about debt,Notification didn't disclose it was an attempt...,There are collection accounts on my report tha...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",LA,70119,NaN,Web,2026-03-24T15:12:46.000Z,Closed with non-monetary relief,Yes,20541178
3,2026-03-24T16:39:24.000Z,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,"These can be combined On my credit report, you...",Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",IN,46410,NaN,Web,2026-03-24T16:45:47.000Z,Closed with non-monetary relief,Yes,20545588
4,2026-03-24T16:48:19.000Z,Debt collection,Other debt,Written notification about debt,Notification didn't disclose it was an attempt...,There are collection accounts on my report tha...,NaN,"EQUIFAX, INC.",NY,11692,NaN,Web,2026-03-24T16:53:56.000Z,Closed with non-monetary relief,Yes,20546051
